In [1]:
import os 
from pathlib import Path 
import pandas as pd 
import numpy as np 

In [2]:
os.chdir("..")

In [3]:
print(os.getcwd())

e:\project_archive\student-dropout-enrolled-graduate-rate-prediction


In [9]:
# Load data

data_path = Path("data/processed")
if not data_path.exists():
    raise FileNotFoundError

X_train = pd.read_csv(data_path / "x_train.csv", sep=';')
y_train = np.load(data_path / "y_train.npy")

X_test = pd.read_csv(data_path / "x_test.csv", sep=';')
y_test = np.load(data_path / "y_test.npy")


In [7]:
X_train.head()

,Marital status,Application mode,Application order,Course,Previous qualification,Previous qualification (grade),Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,...,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Inflation rate
0,4,7,1,9147,3,130.0,19,1,5,5,...,0,1,0,0,35,5,5,0,0.000000,0.6
1,1,39,1,9085,1,130.0,37,37,6,6,...,0,1,0,1,25,6,13,3,11.666667,0.6
2,1,1,6,9070,6,119.0,1,1,9,9,...,0,1,1,0,22,6,6,6,14.166667,1.4
3,2,39,1,9238,19,133.1,37,37,9,4,...,0,1,1,0,42,6,0,0,0.000000,2.8
4,1,1,3,9500,1,142.0,37,38,9,9,...,0,1,0,1,22,7,7,6,13.900000,2.6


In [8]:
y_train

array([0, 1, 2, ..., 2, 2, 0], shape=(3539,))

In [10]:
X_test.head()

,Marital status,Application mode,Application order,Course,Previous qualification,Previous qualification (grade),Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,...,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Inflation rate
0,4,39,1,9130,1,133.1,3,1,5,5,...,0,1,0,1,30,6,7,0,0.000000,0.6
1,1,17,1,9238,1,125.0,4,3,1,1,...,0,1,0,0,18,6,9,5,11.571429,0.3
2,1,17,1,9853,1,133.0,38,38,9,9,...,1,1,0,1,18,7,7,7,12.714286,0.3
3,1,17,2,9670,1,110.0,1,1,4,10,...,0,1,1,0,19,6,8,6,13.857143,2.8
4,1,39,1,9500,1,130.0,37,19,9,8,...,0,1,0,0,27,7,14,0,0.000000,0.6


In [11]:
config = {
    "xgboost": {
        "n_estimators": (200, 1000),
        "learning_rate": (0.01, 0.3),
        "max_depth": (3, 10),
        "min_child_weight": (1, 10),
        "subsample": (0.6, 1.0),
        "colsample_bytree": (0.6, 1.0),
        "gamma": (0.0, 5.0),
        "reg_alpha": (1e-5, 10.0),
        "reg_lambda": (1e-5, 10.0),
    }
}

In [12]:
from xgboost import XGBClassifier


def suggest_params(trial, config):

    cfg = config["xgboost"]

    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators",
            cfg["n_estimators"][0],
            cfg["n_estimators"][1],
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            cfg["learning_rate"][0],
            cfg["learning_rate"][1],
            log=True,
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            cfg["max_depth"][0],
            cfg["max_depth"][1],
        ),

        "min_child_weight": trial.suggest_int(
            "min_child_weight",
            cfg["min_child_weight"][0],
            cfg["min_child_weight"][1],
        ),

        "subsample": trial.suggest_float(
            "subsample",
            cfg["subsample"][0],
            cfg["subsample"][1],
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            cfg["colsample_bytree"][0],
            cfg["colsample_bytree"][1],
        ),

        "gamma": trial.suggest_float(
            "gamma",
            cfg["gamma"][0],
            cfg["gamma"][1],
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            cfg["reg_alpha"][0],
            cfg["reg_alpha"][1],
            log=True,
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            cfg["reg_lambda"][0],
            cfg["reg_lambda"][1],
            log=True,
        ),
    }

    return params

In [13]:
import wandb
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score


def objective(trial, X, y, config):

    params = suggest_params(
        trial=trial,
        config=config,
    )

    model = XGBClassifier(
        **params,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
    )

    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    scores = cross_val_score(
        estimator=model,
        X=X,
        y=y,
        cv=cv,
        scoring="f1_macro",
        n_jobs=-1,
        error_score="raise",
    )

    mean_score = scores.mean()
    std_score = scores.std()

    trial.set_user_attr("cv_std", std_score)

    wandb.log({
        "trial": trial.number,
        "cv_f1_macro": mean_score,
        "cv_f1_macro_std": std_score,
    })

    return mean_score

In [14]:
import pandas as pd
import optuna
import wandb

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

with wandb.init(
    project="student-drop-enroll-grad-preds",
    name="XGBoost-Optuna-v1",
    group="XGBoost",
    tags=["Optuna", "Hyperparameter-Tuning"],
    config={
        "model": "XGBoost",
        "cv": "StratifiedKFold",
        "metric": "f1_macro",
        "n_trials": 80,
        "random_state": 42,
    },
):

    study = optuna.create_study(
        direction="maximize",
        study_name="xgboost_tuning",
    )

    study.optimize(
        lambda trial: objective(
            trial,
            X_train,
            y_train,
            config,
        ),
        n_trials=80,
        show_progress_bar=True,
    )

    best_params = study.best_params

    final_model = XGBClassifier(
        **best_params,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=-1,
    )

    final_model.fit(X_train, y_train)

    predictions = final_model.predict(X_test)

    metrics = {
        "test_accuracy": accuracy_score(y_test, predictions),
        "test_balanced_accuracy": balanced_accuracy_score(y_test, predictions),
        "test_precision_macro": precision_score(
            y_test,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "test_recall_macro": recall_score(
            y_test,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "test_f1_macro": f1_score(
            y_test,
            predictions,
            average="macro",
        ),
        "test_f1_weighted": f1_score(
            y_test,
            predictions,
            average="weighted",
        ),
    }

    wandb.log(metrics)

    wandb.summary["best_cv_f1_macro"] = study.best_value
    wandb.summary["best_trial"] = study.best_trial.number
    wandb.summary["best_params"] = best_params

    for key, value in metrics.items():
        wandb.summary[key] = value

    results = pd.DataFrame(
        {
            "Actual": y_test,
            "Prediction": predictions,
        }
    )

    wandb.log(
        {
            "Predictions": wandb.Table(dataframe=results)
        }
    )

e:\project_archive\student-dropout-enrolled-graduate-rate-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
wandb: Currently logged in as: h41497254 (h41497254-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


[I 2026-07-31 23:24:54,612] A new study created in memory with name: xgboost_tuning
Best trial: 0. Best value: 0.635352:   1%|▏         | 1/80 [00:09<12:25,  9.44s/it]

[I 2026-07-31 23:25:04,045] Trial 0 finished with value: 0.6353520729326216 and parameters: {'n_estimators': 573, 'learning_rate': 0.024872428728885707, 'max_depth': 4, 'min_child_weight': 2, 'subsample': 0.9847780290316084, 'colsample_bytree': 0.89869255858502, 'gamma': 4.26052667934015, 'reg_alpha': 0.02032054811509089, 'reg_lambda': 1.1347651669678895}. Best is trial 0 with value: 0.6353520729326216.


Best trial: 1. Best value: 0.649583:   2%|▎         | 2/80 [00:15<10:00,  7.69s/it]

[I 2026-07-31 23:25:10,520] Trial 1 finished with value: 0.6495834979319832 and parameters: {'n_estimators': 519, 'learning_rate': 0.025673379408092972, 'max_depth': 7, 'min_child_weight': 2, 'subsample': 0.6906493481629568, 'colsample_bytree': 0.8187175899554845, 'gamma': 2.664824461228968, 'reg_alpha': 0.007296426046583347, 'reg_lambda': 0.3450733251120722}. Best is trial 1 with value: 0.6495834979319832.


...

[I 2026-07-31 23:28:08,416] Trial 78 finished with value: 0.6714721874451588 and parameters: {'n_estimators': 643, 'learning_rate': 0.02015404383415536, 'max_depth': 4, 'min_child_weight': 5, 'subsample': 0.9229246547907006, 'colsample_bytree': 0.7409722194133906, 'gamma': 0.6265088355860507, 'reg_alpha': 0.02639680936570852, 'reg_lambda': 0.00021130045225038212}. Best is trial 19 with value: 0.6775877477917189.


Best trial: 19. Best value: 0.677588: 100%|██████████| 80/80 [03:15<00:00,  2.44s/it]


[I 2026-07-31 23:28:09,795] Trial 79 finished with value: 0.6630584662708053 and parameters: {'n_estimators': 391, 'learning_rate': 0.15004438379065144, 'max_depth': 5, 'min_child_weight': 4, 'subsample': 0.9449978913062133, 'colsample_bytree': 0.8633723887861656, 'gamma': 0.3012609511120151, 'reg_alpha': 0.016697212305385286, 'reg_lambda': 0.000930228766634587}. Best is trial 19 with value: 0.6775877477917189.


wandb: ERROR The nbformat package was not found. It is required to save notebook history.


cv_f1_macro,▄▅▄▁▇▇▇▇▇██▆▆▇▅▇█▇▇▇█▇▇▃▇▆▇▆▆▇▇█▇▆█▆█▇▇█
cv_f1_macro_std,▃▁▃▃▅▂▅▅▃▆▆▇▅▄▄▃▃▅▃▆▄▃▄▃▃▄▅█▅▆▃▆▁▄▅▅▄▃▆▇
test_accuracy,▁
test_balanced_accuracy,▁
test_f1_macro,▁
test_f1_weighted,▁
test_precision_macro,▁
test_recall_macro,▁
trial,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
best_cv_f1_macro,0.67759
best_trial,19


In [15]:
study.best_params

{'n_estimators': 696,
 'learning_rate': 0.16157597011642857,
 'max_depth': 3,
 'min_child_weight': 3,
 'subsample': 0.9384557142483714,
 'colsample_bytree': 0.6003312273069528,
 'gamma': 0.6892216422740459,
 'reg_alpha': 0.042248903230999314,
 'reg_lambda': 0.00010645435450077591}

In [16]:
study.best_value

0.6775877477917189

In [17]:
import json 

save_result_path = Path("artifacts/reports")

save_result_path.mkdir(parents=True, exist_ok=True)

result = {
    "model_name": "XGBoost",
    "best_trial": study.best_trial.number,
    "best_params": study.best_params,
    "best_value": float(study.best_value)
}

with open(save_result_path / "xgboost.json", "w") as f:
    json.dump(result, f, indent=4)
    